# Classifier: Cohort A (registry-confirmed) vs. Cohort B (EHR-only)

Trains a binary classifier (XGBoost) to distinguish patient-episodes that were confirmed by the
cancer registry (Cohort A) from EHR-only episodes (Cohort B), using the treatment-intensity and
utilization-density features built in the earlier notebooks. Includes MLflow experiment tracking, a
patient-aware train/validation/test split, threshold tuning, and SHAP-based feature importance.

Reads from `cohort_a_derived.cohort_a_b_feature_table_edi` (built in
`08_union_cohort_a_and_b`).

In [ ]:
# =============================================================================
# Cohort A/B Binary Classifier — Random Forest & XGBoost
# MLflow-integrated | Databricks Single-Node | 70/15/15 Split
# =============================================================================
# Each section below is a separate notebook cell.
# In Databricks, cells are delimited by:  # COMMAND ----------
# =============================================================================

In [ ]:
%sql
USE CATALOG your_catalog;

In [ ]:
# =============================================================================
# CELL 1 — INSTALL / IMPORT DEPENDENCIES
# =============================================================================
# Run this cell first. If any packages are missing from your cluster,
# uncomment the pip install lines below.

# %pip install xgboost scikit-learn matplotlib --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, RocCurveDisplay,
    ConfusionMatrixDisplay,
)
import xgboost as xgb
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature

print("All dependencies loaded successfully.")

In [ ]:
# =============================================================================
# CELL 2 — CONFIGURATION
# =============================================================================
# Edit values here. Nothing else in the notebook should need changing.

TABLE_PATH   = "cohort_a_derived.cohort_a_b_feature_table_edi"
LABEL_COL    = "label"
ID_COL       = "ehr_person_id"
LABEL_POS    = "cohort a reg and ehr"   # Maps to 1 (positive class)
LABEL_NEG    = "cohort b ehr only"      # Maps to 0 (negative class)

FEATURE_COLS = [
    "total_contact_days",
    "years_w_unc",
    "avg_days_bt_visits",
    "rad_days",
    "surg_days",
    "chemproc_days",
    "chemdrug_inst",
    "ccode_days",
    "ccode_ratio",
    "treatment_ratio",
    "gmm_4_cluster",
    "domain_resid_condition", 
    "domain_resid_drug", 
    "domain_resid_measurement", 
    "domain_resid_procedure"
]

# Split ratios
TRAIN_SIZE   = 0.70
VAL_SIZE     = 0.15   # of total; remainder goes to test
RANDOM_STATE = 42

# MLflow settings
MLFLOW_EXPERIMENT_NAME = "/Shared/cohort_ab_classifier"  # Update if needed
XGB_RUN_NAME           = "xgboost_v1"

print("Configuration loaded.")
print(f"  Table        : {TABLE_PATH}")
print(f"  Label column : {LABEL_COL}")
print(f"  Features     : {FEATURE_COLS}")
print(f"  Experiment   : {MLFLOW_EXPERIMENT_NAME}")

In [ ]:
# =============================================================================
# CELL 3 — LOAD DATA FROM HIVE METASTORE
# =============================================================================

print(f"Loading table: {TABLE_PATH} ...")
spark_df = spark.table(TABLE_PATH).select([ID_COL, LABEL_COL] + FEATURE_COLS)
df = spark_df.toPandas()

# Cast all feature columns to float to resolve Decimal type from Hive metastore
df[FEATURE_COLS] = df[FEATURE_COLS].astype(float)

print(f"  Total records loaded : {len(df):,}")
print(f"  Columns              : {list(df.columns)}")
print(f"\nLabel distribution:")
print(df[LABEL_COL].value_counts().to_string())

Drop rows with missing feature or label values before proceeding.

In [ ]:
df = df.dropna(subset=FEATURE_COLS + [LABEL_COL])

In [ ]:
# =============================================================================
# CELL 4 — DATA QUALITY CHECKS
# =============================================================================
# Checks for: missing values, unexpected label values, patient record counts,
# and zero-variance features. Raises on critical issues; warns on others.

issues_found = False

# --- 4a. Missing values ---
print("--- Missing Value Check ---")
missing     = df[FEATURE_COLS].isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct%": missing_pct})
missing_report = missing_report[missing_report["missing_count"] > 0]

if missing_report.empty:
    print("  PASS — No missing values in feature columns.")
else:
    issues_found = True
    print("  FAIL — Missing values detected:")
    print(missing_report.to_string())

# --- 4b. Unexpected label values ---
print("\n--- Label Value Check ---")
expected_labels = {LABEL_POS, LABEL_NEG}
actual_labels   = set(df[LABEL_COL].unique())
unexpected      = actual_labels - expected_labels

if not unexpected:
    print(f"  PASS — Only expected label values found: {actual_labels}")
else:
    issues_found = True
    print(f"  FAIL — Unexpected label values found: {unexpected}")

# --- 4c. Patient record counts (multiple records per patient expected) ---
print("\n--- Patient Record Count ---")
n_patients = df[ID_COL].nunique()
n_records  = len(df)
print(f"  {n_patients:,} unique patients across {n_records:,} records "
      f"(avg {n_records/n_patients:.1f} records/patient)")

# --- 4d. Zero-variance features ---
print("\n--- Zero-Variance Feature Check ---")
zero_var = [c for c in FEATURE_COLS if df[c].nunique() <= 1]
if not zero_var:
    print("  PASS — All features have variance.")
else:
    issues_found = True
    print(f"  WARN — Zero-variance features (consider dropping): {zero_var}")

# --- 4e. Class balance ---
print("\n--- Class Balance ---")
label_counts = df[LABEL_COL].value_counts()
for lbl, cnt in label_counts.items():
    print(f"  {lbl}: {cnt:,} ({cnt/len(df)*100:.1f}%)")
ratio = label_counts.max() / label_counts.min()
print(f"  Imbalance ratio (majority:minority): {ratio:.2f}:1")

# --- Final gate ---
print("\n--- Summary ---")
if issues_found:
    raise ValueError(
        "One or more critical data quality checks failed. "
        "Review output above and resolve before continuing."
    )
else:
    print("  All critical checks passed. Safe to proceed.")

In [ ]:
# =============================================================================
# CELL 5 — PREPROCESSING: ENCODE LABELS & SPLIT DATA (patient-aware)
# =============================================================================
# Splits on unique patient IDs so all records for a given patient
# land in exactly one split — prevents leakage across cancer records.

label_map           = {LABEL_POS: 1, LABEL_NEG: 0}
df["label_encoded"] = df[LABEL_COL].map(label_map)

# --- Patient-level split ---
unique_patients = df[ID_COL].unique()
np.random.seed(RANDOM_STATE)
np.random.shuffle(unique_patients)

n       = len(unique_patients)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_ids = unique_patients[:n_train]
val_ids   = unique_patients[n_train : n_train + n_val]
test_ids  = unique_patients[n_train + n_val:]

train_df = df[df[ID_COL].isin(train_ids)]
val_df   = df[df[ID_COL].isin(val_ids)]
test_df  = df[df[ID_COL].isin(test_ids)]

X_train, y_train = train_df[FEATURE_COLS].astype(float).values, train_df["label_encoded"].values
X_val,   y_val   = val_df[FEATURE_COLS].astype(float).values,   val_df["label_encoded"].values
X_test,  y_test  = test_df[FEATURE_COLS].astype(float).values,  test_df["label_encoded"].values

print("Patient-aware split summary:")
for name, ids, arr in [
    ("Train",      train_ids, y_train),
    ("Validation", val_ids,   y_val),
    ("Test",       test_ids,  y_test),
]:
    pos = arr.sum()
    neg = len(arr) - pos
    print(f"  {name:<12}: {len(arr):>6,} records  |  {len(ids):,} patients  "
          f"|  Class 1: {pos:,}  Class 0: {neg:,}")

# --- Class weights (from training set only) ---
n_neg            = (y_train == 0).sum()
n_pos            = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f"\nClass weight for minority (Cohort A): {scale_pos_weight:.4f}")

In [ ]:
# =============================================================================
# CELL 6 — DESCRIPTIVE STATISTICS BY LABEL
# =============================================================================
# Computes per-feature descriptive stats stratified by cohort label.
# Uses the full dataset (pre-split) so counts reflect all available data.
 
stats_rows = []
for feature in FEATURE_COLS:
    for label_val, label_name in [(1, LABEL_POS), (0, LABEL_NEG)]:
        subset = df[df["label_encoded"] == label_val][feature].dropna()
        stats_rows.append({
            "feature"  : feature,
            "cohort"   : label_name,
            "n"        : len(subset),
            "mean"     : subset.mean(),
            "median"   : subset.median(),
            "std"      : subset.std(),
            "min"      : subset.min(),
            "p25"      : subset.quantile(0.25),
            "p75"      : subset.quantile(0.75),
            "max"      : subset.max(),
        })
 
stats_df = pd.DataFrame(stats_rows)
 
# --- Display wide format: one row per feature/cohort ---
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
 
print("DESCRIPTIVE STATISTICS BY COHORT LABEL")
print("=" * 100)
print(stats_df.to_string(index=False))
 
# --- Also display a pivot for easier side-by-side reading ---
print("\n\nMEAN & MEDIAN — SIDE-BY-SIDE COMPARISON")
print("=" * 80)
for feature in FEATURE_COLS:
    sub = stats_df[stats_df["feature"] == feature]
    row_a = sub[sub["cohort"] == LABEL_POS].iloc[0]
    row_b = sub[sub["cohort"] == LABEL_NEG].iloc[0]
    print(f"\n  {feature}")
    print(f"    {'Cohort':<35} {'N':>7}  {'Mean':>9}  {'Median':>9}  {'Std':>9}  {'Min':>7}  {'Max':>7}")
    print(f"    {'-'*85}")
    for row in [row_a, row_b]:
        print(f"    {row['cohort']:<35} {int(row['n']):>7}  "
              f"{row['mean']:>9.2f}  {row['median']:>9.2f}  "
              f"{row['std']:>9.2f}  {row['min']:>7.2f}  {row['max']:>7.2f}")
 
# --- Visualize distributions: box plots per feature, split by cohort ---
n_features = len(FEATURE_COLS)
fig, axes = plt.subplots(nrows=n_features, ncols=1, figsize=(9, 3.5 * n_features))
 
for ax, feature in zip(axes, FEATURE_COLS):
    data_a = df[df["label_encoded"] == 1][feature].dropna()
    data_b = df[df["label_encoded"] == 0][feature].dropna()
    bp = ax.boxplot(
        [data_a, data_b],
        vert=False,
        patch_artist=True,
        labels=[LABEL_POS, LABEL_NEG],
        medianprops=dict(color="black", linewidth=2),
    )
    bp["boxes"][0].set_facecolor("#4C72B0")
    bp["boxes"][1].set_facecolor("#DD8452")
    ax.set_title(feature, fontsize=11, fontweight="bold")
    ax.set_xlabel("Value")
 
plt.suptitle("Feature Distributions by Cohort", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/tmp/feature_distributions.png", dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)
print("\nDistribution plot saved to /tmp/feature_distributions.png")
 

In [ ]:
# =============================================================================
# CELL 7 — HELPER FUNCTIONS
# =============================================================================

def compute_metrics(y_true, y_pred, y_prob):
    """Return a dict of evaluation metrics."""
    return {
        "auc_roc"   : roc_auc_score(y_true, y_prob),
        "f1"        : f1_score(y_true, y_pred),
        "precision" : precision_score(y_true, y_pred),
        "recall"    : recall_score(y_true, y_pred),
    }


def print_metrics(model_name, split_name, metrics, y_true, y_pred):
    """Pretty-print metrics and classification report."""
    print(f"\n{'='*60}")
    print(f"  {model_name} | {split_name}")
    print(f"{'='*60}")
    for k, v in metrics.items():
        print(f"  {k:<14}: {v:.4f}")
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n  Confusion Matrix:")
    print(f"    TN={cm[0,0]:,}  FP={cm[0,1]:,}")
    print(f"    FN={cm[1,0]:,}  TP={cm[1,1]:,}")
    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred, target_names=[LABEL_NEG, LABEL_POS]))


def make_roc_plot(models_data):
    """
    Plot ROC curves for multiple models on the test set.
    models_data: list of (name, y_true, y_prob) tuples.
    Returns a matplotlib Figure.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    for name, y_true, y_prob in models_data:
        RocCurveDisplay.from_predictions(y_true, y_prob, name=name, ax=ax)
    ax.plot([0, 1], [0, 1], "k--", label="Random baseline")
    ax.set_title("ROC Curve — Test Set", fontsize=13)
    ax.legend(loc="lower right")
    plt.tight_layout()
    return fig


def make_confusion_matrix_plot(model_name, y_true, y_pred):
    """Plot and return a confusion matrix figure."""
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=[LABEL_NEG, LABEL_POS],
        ax=ax,
        colorbar=False,
        cmap="Blues",
    )
    ax.set_title(f"Confusion Matrix — {model_name} (Test Set)", fontsize=11)
    plt.tight_layout()
    return fig


def make_feature_importance_plot(importances, title):
    """Bar chart of feature importances. Returns a Figure."""
    fig, ax = plt.subplots(figsize=(7, 4))
    importances.sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Importance")
    plt.tight_layout()
    return fig


print("Helper functions defined.")

In [ ]:
# =============================================================================
# CELL 9 — TRAIN & LOG XGBOOST WITH MLFLOW
# =============================================================================

XGB_PARAMS = {
    "n_estimators"     : 300,
    "max_depth"        : 6,
    "learning_rate"    : 0.05,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "eval_metric"      : "auc",
    "random_state"     : RANDOM_STATE,
    "n_jobs"           : -1,
}

print(f"Starting MLflow run: {XGB_RUN_NAME}")

with mlflow.start_run(run_name=XGB_RUN_NAME) as xgb_run:

    xgb_run_id = xgb_run.info.run_id

    # --- Train ---
    xgb_model = xgb.XGBClassifier(
        **XGB_PARAMS,
        scale_pos_weight=scale_pos_weight,
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50,
    )

    # --- Predict on val + test ---
    xgb_val_prob  = xgb_model.predict_proba(X_val)[:, 1]
    xgb_val_pred  = xgb_model.predict(X_val)
    xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]
    xgb_test_pred = xgb_model.predict(X_test)

    xgb_val_metrics  = compute_metrics(y_val,  xgb_val_pred,  xgb_val_prob)
    xgb_test_metrics = compute_metrics(y_test, xgb_test_pred, xgb_test_prob)

    # --- Log params ---
    mlflow.log_params({**XGB_PARAMS, "scale_pos_weight": scale_pos_weight})

    # --- Log val + test metrics ---
    mlflow.log_metrics({f"val_{k}":  v for k, v in xgb_val_metrics.items()})
    mlflow.log_metrics({f"test_{k}": v for k, v in xgb_test_metrics.items()})

    # --- Log XGBoost eval history (val AUC per round) ---
    evals_result    = xgb_model.evals_result()
    val_auc_history = evals_result.get("validation_0", {}).get("auc", [])
    for step, auc_val in enumerate(val_auc_history):
        mlflow.log_metric("xgb_val_auc_by_round", auc_val, step=step)

    # --- Log feature importances as artifact ---
    xgb_importances = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
    xgb_fi_path = "/tmp/xgb_feature_importances.csv"
    xgb_importances.sort_values(ascending=False).to_csv(xgb_fi_path, header=["importance"])
    mlflow.log_artifact(xgb_fi_path, artifact_path="feature_importances")

    # --- Log plots ---
    roc_fig = make_roc_plot([("XGBoost", y_test, xgb_test_prob)])
    mlflow.log_figure(roc_fig, "plots/roc_curve.png")
    plt.close(roc_fig)

    cm_fig = make_confusion_matrix_plot("XGBoost", y_test, xgb_test_pred)
    mlflow.log_figure(cm_fig, "plots/confusion_matrix.png")
    plt.close(cm_fig)



    DISPLAY_NAMES = {
     "total_contact_days": "Total Contact Days",
    "years_w_unc": "Years with UNC",
    "avg_days_bt_visits": "Average Days Between Visits",
    "rad_days": "Radiation Days",
    "surg_days": "Cancer-related Surgery Days",
    "chemproc_days": "Chemo Procedure Days",
    "chemdrug_inst": "Chemo Drug Days",
    "ccode_days": "Cancer Coded Days",
    "ccode_ratio": "Cancer Coded Ratio",
    "treatment_ratio": "Treatment Ratio",
    "gmm_4_cluster": "EDI Utilization Cluster",
    "domain_resid_condition": "EDI Condition Residual",
    "domain_resid_measurement": "EDI Measurement Residual",
    "domain_resid_drug": "EDI Drug Residual",
    "domain_resid_procedure": "EDI Procedure Residual",
    }

    xgb_importances = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
    xgb_importances = xgb_importances.rename(index=DISPLAY_NAMES)  
    
    fi_fig = make_feature_importance_plot(xgb_importances, "XGBoost Feature Importances")
    mlflow.log_figure(fi_fig, "plots/feature_importances.png")
    plt.close(fi_fig)


    print("\n  XGBoost Feature Importances:")
    print(xgb_importances.sort_values(ascending=False).to_string())
    display(fi_fig)

    # --- Log model (no registry) ---
    signature = infer_signature(X_train, xgb_model.predict(X_train))
    mlflow.xgboost.log_model(
        xgb_model,
        artifact_path="model",
        signature=signature,
        input_example=pd.DataFrame(X_train[:5], columns=FEATURE_COLS),
    )

    print_metrics("XGBoost", "Validation", xgb_val_metrics, y_val, xgb_val_pred)
    print_metrics("XGBoost", "Test",       xgb_test_metrics, y_test, xgb_test_pred)
    print(f"\nMLflow run ID: {xgb_run_id}")

In [ ]:
# =============================================================================
# CELL 10 — SIDE-BY-SIDE COMPARISON & COMBINED ROC PLOT
# =============================================================================

# --- Summary table ---
summary_rows = [
 
    {"Model": "XGBoost",       "Split": "Validation", **xgb_val_metrics},
    {"Model": "XGBoost",       "Split": "Test",       **xgb_test_metrics},
]
summary_df = pd.DataFrame(summary_rows)
summary_df[["auc_roc","f1","precision","recall"]] = \
    summary_df[["auc_roc","f1","precision","recall"]].round(4)

print("\nFINAL MODEL COMPARISON")
print("=" * 65)
print(summary_df.to_string(index=False))

# --- Combined ROC curve on test set ---
combined_roc = make_roc_plot([
    ("XGBoost",       y_test, xgb_test_prob),
]) #rename later
combined_roc.savefig("/tmp/combined_roc_curve.png", dpi=150)
display(combined_roc)
plt.close(combined_roc)

# --- Log combined ROC to both runs ---
for run_id in [xgb_run_id]:

    with mlflow.start_run(run_id=run_id):
        mlflow.log_artifact("/tmp/combined_roc_curve.png", artifact_path="plots")

print("\nCombined ROC curve logged to both MLflow runs.")

### Model interpretability (SHAP)
Computes SHAP values for the XGBoost model on the test set and ranks features by mean absolute SHAP
value (global feature importance), then visualizes the results.

In [ ]:
import shap




# --- XGBoost ---
xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap_vals = xgb_explainer.shap_values(X_test)
# XGBoost returns shape (n_samples, n_features) for binary,
# or (n_samples, n_features, n_classes) for multiclass

# ── 3. Global feature importance — mean |SHAP| ───────────────────────────────
def mean_abs_shap(shap_vals, feature_names, model_name):
    """Handles both binary and multiclass SHAP output shapes."""
    if shap_vals.ndim == 3:
        # XGB multiclass: (n_samples, n_features, n_classes)
        vals = np.mean(np.abs(shap_vals), axis=(0, 2))
    else:
        # Binary: (n_samples, n_features)
        vals = np.abs(shap_vals)

    importance = pd.DataFrame({
        'feature': feature_names,
        'mean_abs_shap': np.mean(vals, axis=0)
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    importance['model'] = model_name
    return importance



xgb_importance = mean_abs_shap(xgb_shap_vals, FEATURE_COLS, 'XGBoost')


print("\n=== XGB Feature Importance (mean |SHAP|) ===")
print(xgb_importance)



# --- XGB beeswarm ---
import matplotlib.pyplot as plt 

def get_shap_for_plot(shap_vals):
    if isinstance(shap_vals, list): 
        return np.mean([np.abs(sv) for sv in shap_vals], axis = 0)
    elif shap_vals.ndim ==3: 
        return shap_vals[:,:,1]
    return shap_vals 


shap.summary_plot(
    get_shap_for_plot(xgb_shap_vals),
    X_test,
    feature_names=[DISPLAY_NAMES.get(c, c) for c in FEATURE_COLS],
    plot_type='dot', show=False
)
fig = plt.gcf()
fig.axes[0].set_title('XGBoost — SHAP Beeswarm', fontweight='bold')
plt.savefig('shap_beeswarm_xgb.png', dpi=150, bbox_inches='tight')
plt.show()